# 본문 수집 실패 기사 다시 확인 (로컬용)

`본문_bs4_재실패_*.json`에 들어 있는 실패 기사 주소만 크롬 창으로 다시 열어 봄. 성공한 기사는 별도 CSV로 저장하고, 필요하면 기존 `본문_bs4_*.csv`에 합침.

- 입력: `data/본문_bs4_재실패_*.json`
- 출력: `data/본문_selenium_재시도성공_*.csv`, `data/본문_selenium_재시도실패_*.json`
- 특징: 실패 기사만 다시 확인, 크롬 화면 확인 가능, 기존 CSV 백업 후 합치기


In [ ]:
# # 필요한 패키지 설치
# # 로컬 커널에 필요한 도구가 이미 있으면 이 셀은 건너뛰어도 됨
# %pip install -q selenium pandas


In [ ]:
from pathlib import Path
import json
import os
import platform
import random
import shutil
import subprocess
import time

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 기사 주소 수집 때와 같은 브라우저 정보 사용
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'

# 크롬 창을 직접 보면서 확인하려면 False, 창 없이 돌리려면 True로 변경
# 크롬 창이 안 뜨면 True로 바꾸면 됨
HEADLESS = False

# 네이버에 너무 자주 접속하지 않도록 잠깐씩 쉬기
RETRY_PAUSE_RANGE_SEC = (1.0, 2.5)
# 기사 페이지를 연 뒤 본문이 뜰 시간 조금 주기
PAGE_LOAD_WAIT_SEC = 0.8
# 제목이나 본문이 바로 안 보일 때 최대 몇 초까지 기다릴지 설정
SELENIUM_WAIT_SEC = 8

# 성공한 기사를 기존 본문_bs4 CSV에 바로 합침. False면 성공 CSV만 따로 만듦
MERGE_TO_BS4_CSV = True

# 합치기 전 기존 CSV를 한 번 복사해 둠
MAKE_BACKUP = True

# 다시 확인할 실패 파일 직접 지정 가능
# None이면 data 폴더의 모든 본문_bs4_재실패_*.json을 다시 확인함
# 특정 파일만 하려면 예: TARGET_FAILURE_FILES = ['본문_bs4_재실패_SBS_260505_260505.json']
TARGET_FAILURE_FILES = None

# === 기존 자동 탐색 (Colab 또는 Drive 동기화 환경용) — 주석처리 ===
# NOTEBOOK_DIR = Path.cwd()
# if NOTEBOOK_DIR.name != 'crawling':
#     candidate = Path('/home/carol/Text-data-Analysis_26-Spring/notebooks/crawling')
#     NOTEBOOK_DIR = candidate if candidate.exists() else NOTEBOOK_DIR
# DATA_DIR = PROJECT_DIR / 'data' / 'crawling'

# === 현재 로컬 작업 폴더에 직접 지정 ===
# data/crawling/ 아래에 실패 목록과 기존 본문 CSV가 모두 들어 있음
DATA_DIR = Path('/home/carol/Text-data-Analysis_26-Spring/data/crawling')

print(f'DATA_DIR: {DATA_DIR}')
print(f'User-Agent: {USER_AGENT}')


In [ ]:
# 대기 시간을 랜덤으로 주고 화면에 기록
def polite_sleep(label, pause_range=RETRY_PAUSE_RANGE_SEC):
    pause_sec = random.uniform(*pause_range)
    print(f'{label} {pause_sec:.1f}초 대기')
    time.sleep(pause_sec)


# 현재 컴퓨터에 설치된 크롬 위치 찾기
# 크롬 위치를 직접 알려 주면 실행 오류가 줄어듦
def find_chrome_binary():
    candidates = []

    if platform.system() == 'Windows':
        # 윈도우에서 크롬이 자주 설치되는 폴더들을 후보로 둠
        candidates.extend([
            os.path.expandvars(r'%ProgramFiles%\Google\Chrome\Application\chrome.exe'),
            os.path.expandvars(r'%ProgramFiles(x86)%\Google\Chrome\Application\chrome.exe'),
            os.path.expandvars(r'%LocalAppData%\Google\Chrome\Application\chrome.exe'),
        ])
    else:
        # 리눅스나 WSL에서는 크롬 이름 후보를 차례로 확인
        for name in ['google-chrome', 'google-chrome-stable', 'chromium-browser', 'chromium']:
            found = shutil.which(name)
            if found:
                candidates.append(found)

    # 실제로 존재하는 첫 번째 경로 사용
    for path in candidates:
        if path and Path(path).exists():
            return str(Path(path))
    return None


# 실패한 기사 주소를 다시 열 크롬 준비
# 기본값은 사람이 화면을 보면서 확인할 수 있도록 창을 띄우는 방식임
def build_driver(headless=HEADLESS):
    options = Options()
    options.add_argument(f'user-agent={USER_AGENT}')  # 수집할 때 쓰는 기본 정보 맞춤
    options.add_experimental_option('excludeSwitches', ['enable-automation'])  # 크롬이 자동 실행 창처럼 보이는 표시를 줄임
    options.add_experimental_option('useAutomationExtension', False)  # 크롬이 자동 실행 창처럼 보이는 표시를 줄임
    options.add_argument('--disable-blink-features=AutomationControlled')  # 크롬이 자동 실행 창처럼 보이는 표시를 줄임
    options.add_argument('--window-size=1400,1000')  # 항상 비슷한 화면 크기로 열기

    if headless:
        options.add_argument('--headless=new')  # 창 없이 실행

    if platform.system() != 'Windows':
        options.add_argument('--no-sandbox')  # 리눅스/WSL에서 크롬 실행 오류 줄이기
        options.add_argument('--disable-dev-shm-usage')  # 크롬이 중간에 꺼지는 문제 줄이기
        options.add_argument('--disable-gpu')  # 창 없이 실행할 때 그래픽 관련 오류 줄이기

    # 크롬 실행 파일을 찾으면 직접 지정, 못 찾으면 기본 방식으로 실행
    chrome_binary = find_chrome_binary()
    if chrome_binary:
        options.binary_location = chrome_binary
        print(f'Chrome binary: {chrome_binary}')
        subprocess.run([chrome_binary, '--version'], check=False)
    else:
        print('크롬 실행 파일을 직접 찾지 못해 Selenium 기본 방식으로 실행')

    # 현재 크롬에 맞는 실행 도구로 크롬 창 열기
    driver = webdriver.Chrome(service=Service(), options=options)

    # 네이버가 자동 실행 창이라고 판단할 가능성을 조금 낮춤
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
    })
    return driver


driver = build_driver()


In [ ]:
import re


# 스포츠/연예 기사처럼 일반 뉴스와 다른 곳으로 열리는 주소 구분용 표
# 앞에는 주소에 들어 있는 글자, 뒤에는 저장할 카테고리 이름과 세부 카테고리 찾는 규칙 입력
# 나중에 다른 종류가 생기면 이 표에만 추가하면 됨
REDIRECT_DOMAINS = {
    'sports.naver.com': ('스포츠', r'https://m\.sports\.naver\.com/([^/]+)/article/'),
    'entertain.naver.com': ('연예', r'https://m\.entertain\.naver\.com/([^/]+)/article/'),
}


# '2026.05.05. 오전 7:10'처럼 보이는 시간을 '2026-05-05 07:10:00' 모양으로 변경
def parse_korean_datetime(text):
    m = re.match(r'(\d{4})\.(\d{2})\.(\d{2})\.\s*(오전|오후)\s*(\d{1,2}):(\d{2})', text)
    if not m:
        return ''
    y, mo, d, ampm, h, mi = m.groups()
    h = int(h)
    # 오후 시간은 12를 더하고, 오전 12시는 0시로 변경
    if ampm == '오후' and h != 12:
        h += 12
    elif ampm == '오전' and h == 12:
        h = 0
    return f'{y}-{mo}-{d} {h:02d}:{mi}:00'


# 스포츠/연예 기사는 일반 뉴스와 제목·본문·날짜 위치가 달라 따로 처리
def extract_redirected_article_selenium(driver, original_link):
    current_url = driver.current_url

    # 페이지 정보에 들어 있는 제목 가져오기
    title_elements = driver.find_elements(By.CSS_SELECTOR, 'meta[property="og:title"]')
    title = title_elements[0].get_attribute('content').strip() if title_elements else ''

    # 스포츠/연예 기사 본문 영역 가져오기
    body_elements = driver.find_elements(By.CSS_SELECTOR, 'div._article_content')
    body = body_elements[0].text.replace('\n', '').strip() if body_elements else ''

    # 날짜가 여러 개일 수 있어 첫 번째 날짜를 입력일로 사용
    date_elements = driver.find_elements(By.CSS_SELECTOR, 'em.date')
    pubdate = parse_korean_datetime(date_elements[0].text.strip()) if date_elements else ''

    # 주소에 sports나 entertain이 들어 있는지 보고 카테고리 정리
    # 예: m.sports.naver.com/golf/article/... -> '스포츠/golf'
    category = '기타'
    for domain, (prefix, pattern) in REDIRECT_DOMAINS.items():
        if domain in current_url:
            cat_match = re.match(pattern, current_url)
            category = f'{prefix}/{cat_match.group(1)}' if cat_match else prefix
            break

    # 제목, 본문, 날짜 중 하나라도 없으면 실패로 남김
    if not title or not body or not pubdate:
        raise ValueError(f'다른 형식 기사 추출 실패: title={bool(title)}, body={bool(body)}, pubdate={bool(pubdate)}')

    # 저장할 때는 처음 실패 목록에 있던 기사 주소 그대로 남김
    return {
        'link': original_link,
        'pubdate': pubdate,
        'category': category,
        'title': title,
        'body': body,
    }


# 기사 한 건에서 제목, 본문, 날짜, 카테고리 가져오기
# 처음에 바로 안 보이면 한 번 더 기다렸다가 다시 확인
def extract_article_selenium(driver, link):
    # 실제 네이버 뉴스 페이지 열기
    driver.get(link)

    # 페이지가 열릴 시간 조금 주기
    wait = WebDriverWait(driver, SELENIUM_WAIT_SEC)
    time.sleep(PAGE_LOAD_WAIT_SEC)

    # 스포츠/연예 기사처럼 다른 형식으로 열리면 별도 함수로 처리
    if any(domain in driver.current_url for domain in REDIRECT_DOMAINS):
        return extract_redirected_article_selenium(driver, link)

    # 제목 위치를 찾아 제목 가져오기
    title_elements = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
    title = title_elements[0].text.strip() if title_elements else ''

    # 본문 위치를 찾아 줄바꿈을 없앤 뒤 한 문장처럼 정리
    body_elements = driver.find_elements(By.ID, 'newsct_article')
    body = body_elements[0].text.replace('\n', '').strip() if body_elements else ''

    # 기사 날짜는 화면 글자보다 컴퓨터가 읽기 쉬운 날짜 값 사용
    pubdate_elements = driver.find_elements(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')
    pubdate = pubdate_elements[0].get_attribute('data-date-time') if pubdate_elements else ''

    # 페이지 상단에서 선택된 카테고리 이름 가져오기
    category_elements = driver.find_elements(By.CSS_SELECTOR, 'a.Nitem_link[aria-selected="true"] span.Nitem_link_menu')
    category = category_elements[0].text.strip() if category_elements else ''

    if not title or not body or not pubdate:
        # 본문이 늦게 뜨는 경우가 있어 짧게 기다린 뒤 한 번 더 확인
        wait.until(lambda d: d.find_elements(By.ID, 'newsct_article') or d.find_elements(By.CLASS_NAME, 'media_end_head_headline'))
        title_elements = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
        body_elements = driver.find_elements(By.ID, 'newsct_article')
        pubdate_elements = driver.find_elements(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')

        # 새로 찾은 값이 있으면 그 값으로 변경
        title = title_elements[0].text.strip() if title_elements else title
        body = body_elements[0].text.replace('\n', '').strip() if body_elements else body
        pubdate = pubdate_elements[0].get_attribute('data-date-time') if pubdate_elements else pubdate

    # 제목, 본문, 날짜 중 하나라도 없으면 어느 값이 비었는지 표시하고 실패로 남김
    if not title or not body or not pubdate:
        raise ValueError(f'title/body/pubdate 추출 실패: title={bool(title)}, body={bool(body)}, pubdate={bool(pubdate)}')

    return {
        'link': link,
        'pubdate': pubdate,
        'category': category,
        'title': title,
        'body': body,
    }


In [ ]:
# 재시도 대상 실패 파일 로드
# BS4 수집 단계에서 떨어뜨린 본문_bs4_재실패_*.json 목록을 추려 반환
def load_failure_files(data_dir=DATA_DIR):
    # TARGET_FAILURE_FILES가 지정되면 해당 파일만, 아니면 모든 재실패 파일 처리
    if TARGET_FAILURE_FILES:
        paths = [data_dir / name for name in TARGET_FAILURE_FILES]
    else:
        paths = sorted(data_dir.glob('본문_bs4_재실패_*.json'))

    # 없는 파일은 화면에만 알려 주고 나머지는 계속 진행
    existing = [p for p in paths if p.exists()]
    missing = [p for p in paths if not p.exists()]

    if missing:
        print('찾지 못한 실패 파일:')
        for p in missing:
            print(f'- {p}')

    if not existing:
        raise FileNotFoundError('재시도할 본문_bs4_재실패_*.json 파일이 없습니다.')

    # 파일마다 다시 확인할 기사 수를 미리 보여 줌
    print(f'재시도 대상 실패 파일: {len(existing)}개')
    for p in existing:
        with p.open('r', encoding='utf-8') as f:
            data = json.load(f)
        print(f'- {p.name}: {len(data.get("links", []))}건')
    return existing


failure_files = load_failure_files()


In [ ]:
# 실패 기사 주소만 크롬으로 다시 열어 본문 수집 재시도
# 성공한 기사는 따로 저장하고, 설정에 따라 기존 본문 CSV에도 합침
def retry_failure_file(failure_path, driver=driver, data_dir=DATA_DIR):
    # 파일명에서 언론사와 기간 부분을 뽑아 관련 파일 경로 생성
    stem = failure_path.stem.replace('본문_bs4_재실패_', '')
    bs4_csv_path = data_dir / f'본문_bs4_{stem}.csv'
    success_path = data_dir / f'본문_selenium_재시도성공_{stem}.csv'
    remain_fail_path = data_dir / f'본문_selenium_재시도실패_{stem}.json'

    # 재실패 파일에서 원래 번호와 기사 주소 목록 불러오기
    with failure_path.open('r', encoding='utf-8') as f:
        failure_data = json.load(f)

    failed_indices = failure_data.get('err_idx', [])
    failed_links = failure_data.get('links', [])

    successes = []
    failures = []

    print()
    print(f'=== {stem} Selenium 재시도 시작: {len(failed_links)}건 ===')

    # 실패한 기사 주소를 하나씩 다시 방문
    # original_idx는 처음 본문 수집 때 몇 번째 기사였는지 확인하려고 남겨 둠
    for seq, (original_idx, link) in enumerate(zip(failed_indices, failed_links), start=1):
        try:
            article = extract_article_selenium(driver, link)
            article['original_idx'] = original_idx
            successes.append(article)
            print(f'성공 [{seq}/{len(failed_links)}] index={original_idx}')
        except Exception as exc:
            # 실패 이유를 같이 저장해 나중에 사람이 확인할 수 있게 함
            failures.append({'original_idx': original_idx, 'link': link, 'error': repr(exc)})
            print(f'실패 [{seq}/{len(failed_links)}] index={original_idx}: {exc!r}')

        polite_sleep('다음 재시도 전')

    # 다시 확인해서 성공한 기사만 별도 CSV로 저장
    success_df = pd.DataFrame(successes)
    if not success_df.empty:
        success_df.to_csv(success_path, index=False, encoding='utf-8-sig')
        print(f'Selenium 성공분 저장: {success_path}')

    # 다시 확인해도 실패한 기사와 실패 이유 저장
    with remain_fail_path.open('w', encoding='utf-8') as f:
        json.dump({'failures': failures}, f, ensure_ascii=False, indent=2)
    print(f'Selenium 재시도 실패 목록 저장: {remain_fail_path}')

    # 성공한 기사를 기존 BS4 CSV에 합침
    if MERGE_TO_BS4_CSV and not success_df.empty:
        if not bs4_csv_path.exists():
            raise FileNotFoundError(f'병합할 기존 CSV가 없습니다: {bs4_csv_path}')

        # 기존 CSV와 성공한 기사를 합친 뒤, 같은 기사 주소는 하나만 남기고 날짜순 정리
        original_df = pd.read_csv(bs4_csv_path, encoding='utf-8-sig')
        # 최종 CSV에 필요한 열만 골라 합침. original_idx는 확인용이라 결과에는 넣지 않음
        merge_df = success_df[['link', 'pubdate', 'category', 'title', 'body']].copy()
        merged_df = pd.concat([original_df, merge_df], ignore_index=True)
        # 같은 기사 주소가 있으면, 뒤에 붙인 재시도 성공분을 남김
        merged_df = merged_df.drop_duplicates(subset=['link'], keep='last').reset_index(drop=True)
        # 날짜순 정렬을 위해 날짜 글자를 날짜 값으로 변경
        merged_df['pubdate'] = pd.to_datetime(merged_df['pubdate'], errors='coerce')
        merged_df = merged_df.sort_values('pubdate').reset_index(drop=True)

        # 합치기 전 기존 CSV를 한 번 복사해 둠. 이미 복사본이 있으면 새로 만들지 않음
        if MAKE_BACKUP:
            backup_path = bs4_csv_path.with_suffix('.csv.bak_before_selenium_retry')
            if not backup_path.exists():
                shutil.copy2(bs4_csv_path, backup_path)
                print(f'기존 CSV 백업: {backup_path}')

        # 정리한 결과를 기존 본문 CSV 위치에 저장
        merged_df.to_csv(bs4_csv_path, index=False, encoding='utf-8-sig')
        print(f'기존 CSV 병합 완료: {bs4_csv_path}')
        print(f'행 수: {len(original_df)} -> {len(merged_df)}')

    print(f'완료: 성공 {len(successes)}건 / 실패 {len(failures)}건')
    return {
        'period': stem,
        'success': len(successes),
        'fail': len(failures),
        'success_path': str(success_path),
        'remain_fail_path': str(remain_fail_path),
    }


In [ ]:
# 생성된 failure_files를 순서대로 재시도
# 한 파일이 실패해도 결과를 요약할 수 있게 파일별 결과 저장
retry_results = []
for failure_path in failure_files:
    result = retry_failure_file(failure_path)
    retry_results.append(result)

# 파일별 성공/실패 수를 표로 확인
summary_df = pd.DataFrame(retry_results)
summary_df


In [ ]:
# 작업이 끝나면 크롬 창 종료
# 필요하면 이 셀만 따로 실행
try:
    driver.quit()
    print('브라우저 종료 완료')
except Exception as exc:
    print(f'브라우저 종료 중 오류: {exc!r}')
